## 3.5: INSERT OVERWRITE con particiones


In [0]:
%sql

-----------------------------
-- CREAMOS TABLA PARTICIONADA
-----------------------------

DROP TABLE IF EXISTS propiedades.silver.ventas_particionada;

CREATE TABLE propiedades.silver.ventas_particionada 
(
    id_venta STRING,
    producto STRING,
    cantidad INT,
    precio_unitario DECIMAL(10,2),
    total DECIMAL(10,2),
    cliente_id STRING,
    fecha_venta DATE
) 
USING DELTA
PARTITIONED BY (fecha_venta)          -- Campo de particionamiento


In [0]:
%sql
--------------------------------------
-- INSERTAMOS ventas en la particion: '2024-01-15'
--------------------------------------

INSERT OVERWRITE propiedades.silver.ventas_particionada

SELECT  id_venta, producto, 
        cantidad, precio_unitario,
        cantidad * precio_unitario as total, 
        cliente_id, fecha_venta

FROM    propiedades.bronze.ventas
WHERE fecha_venta = '2024-01-15';


--------------------------------------
-- Validamos
--------------------------------------
SELECT  fecha_venta, COUNT(*) as filas

FROM    propiedades.silver.ventas_particionada
GROUP BY fecha_venta;

In [0]:
%sql

-----------------------------------------------------------------------
-- Re-ejecutar: la partición 2024-01-15 sigue con las mismas filas
-----------------------------------------------------------------------

INSERT OVERWRITE propiedades.silver.ventas_particionada
SELECT  id_venta, producto, 
        cantidad, precio_unitario,
        cantidad * precio_unitario as total, 
        cliente_id, fecha_venta

FROM    propiedades.bronze.ventas
WHERE   fecha_venta = '2024-01-15';


INSERT INTO propiedades.silver.ventas_particionada
SELECT  id_venta, producto, 
        cantidad, precio_unitario,
        cantidad * precio_unitario as total, 
        cliente_id, fecha_venta

FROM    propiedades.bronze.ventas
WHERE   fecha_venta = '2024-01-16';


--------------------------------------------
-- Validamos que solo haya datos para la fecha
--------------------------------------------
SELECT  fecha_venta, 
        COUNT(*) as filas
FROM    propiedades.silver.ventas_particionada

GROUP BY fecha_venta;

In [0]:
%sql
SHOW PARTITIONS propiedades.silver.ventas_particionada;

In [0]:
%sql
DESCRIBE DETAIL propiedades.silver.ventas_particionada;